# 260402 하이브리드 검색 (Hybrid Search): 키워드 + 벡터 검색의 결합

**4주차 Day 3** | 모두의연구소 재직자 LLM 6기

---

### 오늘 배울 내용
- **키워드 검색** vs **벡터 검색** 비교 -- 각각의 장단점
- **TF-IDF**: 단어 빈도 기반의 똑똑한 키워드 검색
- **BM25**: TF-IDF의 한계를 보완한 실전 키워드 검색 알고리즘
- **하이브리드 검색**: 키워드 + 벡터 검색을 합쳐서 더 나은 결과 얻기
- **LangChain의 EnsembleRetriever**로 하이브리드 검색 구현

### 핵심 비유
> 벡터 검색은 '의미로 찾는 도서관 사서', 키워드 검색은 '정확한 제목으로 찾는 검색 엔진'.
> 둘을 합치면 의미도 잡고, 정확한 용어도 놓치지 않는 **하이브리드 검색**이 된다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w4_rag_evaluation/llm_260402_embeddings_similarity.ipynb)

---
## Step 0: 환경 설정

Colab에서는 아래 설치 셀을 실행하고, Secrets에 `OPENAI_API_KEY`를 등록해주세요.  
로컬에서는 `.env` 파일에 키를 넣고 `load_dotenv()`를 사용합니다.

In [1]:
# ════════════════════════════════════════════════════════════════
# Colab 환경 패키지 설치
# ════════════════════════════════════════════════════════════════
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu # python-dotenv numpy
!pip install -q rank-bm25 sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

---
## Step 1: 임베딩 모델 초기화 & 샘플 문서 준비

10개의 AI/프로그래밍 관련 문장을 준비합니다.  
이 문서들로 키워드 검색, 벡터 검색, 하이브리드 검색을 모두 실험해볼 예정입니다.

In [3]:
import os
import math
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from dotenv import load_dotenv

# load_dotenv()

# OpenAI의 text-embedding-3-small 모델로 임베딩 객체 생성
# (유료 모델 -- 호출마다 비용 발생. 무료 대안은 아래 sentence-transformers 참고)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 검색 실험용 샘플 문서 10개
documents = [
    "Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.",
    "자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니다.",
    "GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행합니다.",
    "RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다.",
    "FastAPI는 Python으로 빠른 웹 API를 구축하기 위한 프레임워크입니다.",
    "트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 모델 구조입니다.",
    "FAISS는 Facebook AI가 개발한 효율적인 유사도 검색 라이브러리입니다.",
    "프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.",
    "임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.",
]

---
## Step 2: 문서 임베딩 생성 (OpenAI vs 오픈소스)

임베딩 = 텍스트를 숫자 벡터로 변환하는 것.  
OpenAI 모델은 1536차원, 오픈소스 MiniLM은 384차원 -- 차원이 다르지만 원리는 같습니다.

In [5]:
# embed_documents(): 여러 문서를 한꺼번에 벡터로 변환
# 반환값: list of list (파이썬 리스트) -- .shape 사용 불가!
doc_embeddings = embeddings.embed_documents(documents)
print(f"문서 수: {len(doc_embeddings)}, 각 벡터 차원: {len(doc_embeddings[0])}")

문서 수: 10, 각 벡터 차원: 1536


In [6]:
# 리스트를 numpy 배열로 변환하면 .shape 사용 가능
# 비유: 리스트는 '장바구니', numpy 배열은 '정리된 엑셀 표'
np.array(doc_embeddings).shape  # (10, 1536) = 10개 문서 x 1536차원

(10, 1536)

In [7]:
# ════════════════════════════════════════════════════════════════
# [참고] 오픈소스 임베딩 모델 -- sentence-transformers
# 장점: 무료, 로컬에서 실행 가능
# 단점: OpenAI보다 성능이 약간 낮을 수 있음
# ════════════════════════════════════════════════════════════════
# !pip install sentence-transformers  # 이미 위에서 설치함

from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2: 가볍고 빠른 영어 임베딩 모델 (384차원)
# HuggingFace에서 모델 카드를 보면 차원, 성능 등을 확인할 수 있음
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding = embedding_model.encode(documents)  # numpy 배열로 바로 반환됨!
print(f"Shape: {embedding.shape}")  # (10, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape: (10, 384)


---
## Step 3: 키워드 검색 (Keyword Search)

가장 단순한 검색: 쿼리 단어와 문서 단어가 **정확히 일치**하는지 확인.  
- 장점: 빠르고, 고유명사/전문용어(약 이름, 법조문 등)에 강함  
- 단점: '파이썬'과 '파이썬은' 을 다른 단어로 인식 (한글 조사 문제)

In [9]:
def keyword_search(query, docs, top_k=3):
    """단순 키워드 매칭 검색"""
    # 1) 쿼리를 소문자로 바꾸고 공백 기준으로 분리 -> set으로 중복 제거
    #    예: "Python 프로그래밍 언어" -> {"python", "프로그래밍", "언어"}
    query_tokens = set(query.lower().split())
    scores = []

    for i, doc in enumerate(docs):
        # 2) 각 문서도 동일하게 토큰화
        doc_tokens = set(doc.lower().split())
        # 3) 교집합(&) = 쿼리와 문서에 공통으로 있는 단어 수
        overlap = len(query_tokens & doc_tokens)
        scores.append((i, overlap))

    # 4) 겹치는 단어가 많은 순서로 정렬
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# 테스트: "Python 프로그래밍 언어"로 검색
results = keyword_search("Python 프로그래밍 언어", documents)
print(results)
for idx, score in results:
    print(f"[{idx}] overlap = {score} | {documents[idx][:40]}")

[(0, 1), (3, 1), (1, 0)]
[0] overlap = 1 | Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니
[3] overlap = 1 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행
[1] overlap = 0 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니


---
## Step 4: 벡터 검색 (Vector Search)

임베딩 공간에서 **코사인 유사도**로 의미적 유사성을 측정합니다.  
코사인 유사도 = (A . B) / (||A|| * ||B||) -- 두 벡터의 각도가 작을수록 1에 가까움.

In [10]:
def vector_search(query, docs, doc_embs, top_k=3):
    """코사인 유사도 기반 벡터 검색"""
    # 1) 쿼리도 벡터로 변환 (embed_query: 단일 텍스트용)
    q_emb = np.array(embeddings.embed_query(query))

    # 2) 코사인 유사도 계산: 내적(dot) / (문서 벡터 크기 * 쿼리 벡터 크기)
    #    np.dot(doc_embs, q_emb) : 각 문서 벡터와 쿼리 벡터의 내적
    #    np.linalg.norm()        : 벡터의 크기(norm, 길이)
    similarities = np.dot(doc_embs, q_emb) / (
        np.linalg.norm(doc_embs, axis=1) * np.linalg.norm(q_emb)
    )

    # 3) argsort: 값이 아니라 '인덱스'를 정렬
    #    [::-1] : 내림차순 (유사도 높은 것부터)
    #    [:top_k] : 상위 k개만
    top_indices = similarities.argsort()[::-1][:top_k]
    return [(i, similarities[i]) for i in top_indices]

# 테스트: 같은 쿼리로 벡터 검색
results = vector_search("Python 프로그래밍 언어", documents, np.array(doc_embeddings), top_k=3)
for idx, score in results:
    print(f"[{idx}] similarity = {score:.4f} | {documents[idx][:40]}")

[0] similarity = 0.5528 | Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니
[1] similarity = 0.3718 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니
[3] similarity = 0.3604 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행


In [11]:
# 벡터 검색이 잘 되는 예: 'FAISS'처럼 유명한 용어는 임베딩 공간에 잘 표현됨
# 하지만 아주 드문 고유명사(신약 이름, 법조문 번호)는 벡터 검색이 약할 수 있음
results = vector_search('FAISS', documents, np.array(doc_embeddings), top_k=3)
for idx, score in results:
    print(f"[{idx}] similarity = {score:.4f} | {documents[idx][:40]}")

[7] similarity = 0.5279 | FAISS는 Facebook AI가 개발한 효율적인 유사도 검색 라이브러
[4] similarity = 0.1923 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다
[5] similarity = 0.0964 | FastAPI는 Python으로 빠른 웹 API를 구축하기 위한 프레임워


---
## Step 5: 키워드 vs 벡터 검색 결과 비교 (Overlap Rate)

두 검색 방법의 결과가 얼마나 겹치는지 확인합니다.  
겹침이 적다 = 서로 다른 문서를 찾아냄 = **합치면 더 좋은 결과**를 기대할 수 있다!

In [12]:
def overlap_rate(keyword_results, vector_results):
    """키워드 검색과 벡터 검색 결과의 겹치는 비율 (자카드 유사도)"""
    # 각 결과에서 문서 인덱스만 추출하여 set으로 변환
    kw_ids = set(idx for idx, _ in keyword_results)
    vec_ids = set(idx for idx, _ in vector_results)

    overlap = kw_ids & vec_ids   # 교집합: 둘 다 찾은 문서
    union = kw_ids | vec_ids     # 합집합: 어느 쪽이든 찾은 문서
    return len(overlap) / len(union)  # 겹치는 비율 (0~1)

# 여러 쿼리에 대해 overlap 비교
for query in ['Python 프로그래밍', '딥러닝 모델', 'FAISS 라이브러리']:
    kw = keyword_search(query, documents, top_k=5)
    vec = vector_search(query, documents, np.array(doc_embeddings), top_k=5)
    rate = overlap_rate(kw, vec)
    print(f"{query} overlap: {rate:.2%}")
    # overlap이 낮을수록 -> 두 방법이 서로 다른 문서를 찾음 -> 합치면 시너지!

Python 프로그래밍 overlap: 66.67%
딥러닝 모델 overlap: 42.86%
FAISS 라이브러리 overlap: 25.00%


---
## Step 6: 심플 하이브리드 검색 (Simple Hybrid Search)

키워드 검색 점수 + 벡터 검색 점수를 합산하는 가장 기본적인 하이브리드 방식.  
**주의**: 두 점수의 스케일이 다르므로 (키워드=정수, 벡터=0~1) **정규화**(max로 나누기)가 필요합니다.

> 비유: 수학 시험(100점 만점)과 영어 시험(50점 만점)을 합산하려면  
> 각각 최고점으로 나눠서 비율(0~1)로 맞춘 뒤 더해야 공정합니다.

In [13]:
def simple_hybrid(query, docs, doc_embs, top_k=3):
    """키워드 + 벡터 점수를 정규화해서 합산하는 하이브리드 검색"""
    # 1) 전체 문서에 대해 키워드/벡터 검색 수행 (top_k = 전체)
    kw = keyword_search(query, docs, top_k=len(docs))
    vec = vector_search(query, docs, np.array(doc_embs), top_k=len(docs))

    # 2) 딕셔너리로 변환: {문서인덱스: 점수}
    kw_scores = {idx: score for idx, score in kw}
    vec_scores = {idx: score for idx, score in vec}

    # 3) 정규화: 각각의 최댓값으로 나눠서 0~1 범위로 맞춤
    #    max값이 0이면 나눗셈 에러 방지를 위해 1로 대체
    kw_max = max(kw_scores.values()) or 1
    vec_max = max(vec_scores.values()) or 1

    # 4) 정규화된 점수를 합산
    combined = {}
    for idx in range(len(docs)):
        kw_score = kw_scores.get(idx, 0) / kw_max     # 키워드 점수 (0~1)
        vec_score = vec_scores.get(idx, 0) / vec_max   # 벡터 점수 (0~1)
        combined[idx] = kw_score + vec_score            # 단순 합산 (0~2)

    # 5) 합산 점수 기준으로 내림차순 정렬 -> top_k개 반환
    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# 테스트: 세 가지 쿼리로 하이브리드 검색
for query in ['Python 프로그래밍 언어', '딥러닝 모델 구조', 'FAISS']:
    results = simple_hybrid(query, documents, np.array(doc_embeddings), top_k=3)
    for idx, score in results:
        print(f"[{idx}] {score:.4f} | {documents[idx][:40]}")
    print("-" * 60)

[0] 2.0000 | Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니
[3] 1.6520 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행
[1] 0.6725 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니
------------------------------------------------------------
[6] 2.0000 | 트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 모델 구조입니다.
[3] 0.4905 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행
[0] 0.4188 | Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니
------------------------------------------------------------
[7] 1.0000 | FAISS는 Facebook AI가 개발한 효율적인 유사도 검색 라이브러
[4] 0.3644 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다
[5] 0.1827 | FastAPI는 Python으로 빠른 웹 API를 구축하기 위한 프레임워
------------------------------------------------------------


---
## Step 7: TF-IDF -- 키워드 검색의 기초 알고리즘

단순 키워드 매칭보다 훨씬 스마트한 방법!

| 항목 | 의미 | 공식 |
|------|------|------|
| **TF** (Term Frequency) | 해당 문서에서 단어가 얼마나 자주 나오는지 | f(t,d) / \|d\| |
| **IDF** (Inverse Doc Frequency) | 전체 문서에서 그 단어가 얼마나 희귀한지 | log(N / 단어가 등장한 문서 수) |

> **비유**: TF는 '한 교실에서 누가 손을 많이 드는지', IDF는 '전교에서 그 학생이 얼마나 특별한지'.  
> 매번 손드는 학생(흔한 단어 '은/는/이/가')은 IDF가 낮아서 중요도가 떨어지고,  
> 특정 수업에서만 손드는 학생(희귀 키워드 'Python')은 IDF가 높아서 핵심 키워드로 인식됩니다.

### TF-IDF 조합 해석
| TF | IDF | 의미 |
|----|-----|------|
| 높음 | 높음 | **핵심 키워드** -- 이 문서에서 자주 나오고, 다른 문서에선 희귀 |
| 높음 | 낮음 | 흔한 단어 (은/는/이/가, the/a) -- 무시해도 됨 |
| 낮음 | 높음 | 희귀하지만 잘 안 나옴 -- 별로 관련 없음 |
| 낮음 | 낮음 | 의미 없는 단어 |

In [14]:
import math

class TFIDF:
    def __init__(self, documents):
        self.docs = documents
        # 각 문서를 소문자로 변환 후 공백 기준 토큰화
        self.tokenized = [doc.lower().split() for doc in documents]
        self.N = len(documents)  # 전체 문서 수

        # df (document frequency): 각 단어가 몇 개 문서에 등장하는지
        self.df = {}
        for tokens in self.tokenized:
            for t in set(tokens):  # set: 한 문서 내 중복 단어 제거 (문서 단위 카운트)
                self.df[t] = self.df.get(t, 0) + 1

    def tf(self, term, doc_tokens):
        """TF: 해당 문서에서 단어의 등장 비율"""
        return doc_tokens.count(term) / len(doc_tokens)

    def idf(self, term):
        """IDF: 전체 문서 대비 희귀도 (높을수록 희귀한 단어)"""
        # df가 0이면 나눗셈 에러 방지를 위해 1로 대체
        return math.log(self.N / self.df.get(term, 1))

    def score(self, query, doc_idx):
        """쿼리의 각 단어에 대해 TF*IDF를 계산하고 합산"""
        tokens = self.tokenized[doc_idx]
        return sum(self.tf(t, tokens) * self.idf(t) for t in query.lower().split())

    def search(self, query, top_k=3):
        """모든 문서에 대해 점수를 계산하고 상위 k개 반환"""
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

# 테스트
tfidf = TFIDF(documents)
for idx, score in tfidf.search('Python 프로그래밍'):
    print(f"[{idx}] {score:.4f} | {documents[idx][:40]}")

[0] 0.2878 | Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니
[1] 0.0000 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니
[2] 0.0000 | 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니


---
## Step 8: BM25 -- TF-IDF의 업그레이드 버전

TF-IDF의 두 가지 한계를 보완한 알고리즘:

1. **TF 포화(saturation)**: 단어가 5번 나오든 500번 나오든 비슷하게 중요한데,  
   TF는 선형으로 증가 -> **k1 파라미터**로 상한선을 설정 (k1+1이 최댓값)

2. **문서 길이 보정**: 1000단어 문서에서 5번 = 밀도 낮음, 100단어에서 5번 = 밀도 높음  
   -> **b 파라미터**로 평균 문서 길이 대비 보정 (b=0: 무시, b=1: 완전 보정)

> 비유: TF-IDF가 '원시적인 저울'이라면, BM25는 '체중계에 체지방률 보정까지 하는 스마트 저울'.

```
BM25 = IDF(t) * [ f(t,d) * (k1 + 1) ] / [ f(t,d) + k1 * (1 - b + b * |d| / avgdl) ]
```
- k1 (보통 1.2~2.0): TF 포화 조절. 클수록 TF 영향이 더 커짐
- b (보통 0.75): 문서 길이 보정 강도. 0이면 무시, 1이면 완전 보정

In [15]:
class BM25:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.k1, self.b = k1, b  # 하이퍼파라미터 (실험으로 최적값 찾기)
        self.docs = documents
        self.tokenized = [doc.lower().split() for doc in documents]
        self.N = len(documents)
        # avgdl: 평균 문서 길이 (문서 길이 보정에 사용)
        self.avgdl = sum(len(d) for d in self.tokenized) / self.N

        # df 계산 (TF-IDF와 동일)
        self.df = {}
        for tokens in self.tokenized:
            for t in set(tokens):
                self.df[t] = self.df.get(t, 0) + 1

    def idf(self, term):
        """BM25용 IDF: 기본 TF-IDF IDF에 평활화(smoothing) 적용"""
        df = self.df.get(term, 0)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def score(self, query, doc_idx):
        """BM25 점수 계산: TF 포화 + 문서 길이 보정이 핵심"""
        tokens = self.tokenized[doc_idx]
        dl = len(tokens)  # 현재 문서 길이
        tf_counter = Counter(tokens)  # 각 단어별 빈도 카운트
        total = 0.0

        for t in query.lower().split():
            tf = tf_counter.get(t, 0)
            # 분자: TF * (k1 + 1) -> TF가 아무리 커도 k1+1에 수렴
            numerator = tf * (self.k1 + 1)
            # 분모: TF + k1 * (문서길이 보정)
            #   (1 - b + b * dl/avgdl): 문서가 평균보다 길면 페널티, 짧으면 보너스
            denominator = tf + (self.k1 * (1 - self.b + self.b * dl / self.avgdl))
            total += self.idf(t) * numerator / denominator

        return total

    def search(self, query, top_k=3):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

# 테스트
bm25 = BM25(documents)
for idx, score in bm25.search('Python 프로그래밍'):
    print(f"[{idx}] {score:.4f} | {documents[idx][:40]}")

[0] 2.0361 | Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니
[1] 0.0000 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니
[2] 0.0000 | 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니


---
## Step 9: LangChain으로 BM25 & 벡터 검색 사용하기

직접 구현 대신 LangChain의 **BM25Retriever**와 **FAISS Retriever**를 사용합니다.  
실전에서는 이 라이브러리를 사용하되, 위의 원리를 이해하고 있으면 디버깅이 쉬워집니다.

In [16]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# 문자열 리스트를 LangChain Document 객체 리스트로 변환
# page_content: 문서 내용, metadata: 부가 정보 (여기선 인덱스)
docs_lc = [
    Document(page_content=d, metadata={"index": i})
    for i, d in enumerate(documents)
]

# BM25Retriever 초기화 -- from_documents()로 문서 전달
bm25_retriever = BM25Retriever.from_documents(docs_lc)
bm25_retriever.k = 3  # 반환할 문서 수

# 검색 실행
results = bm25_retriever.invoke('Python 프로그래밍')
for doc in results:
    print(f"[{doc.metadata['index']}] {doc.page_content[:50]}")

[0] Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.
[9] 임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.
[8] 프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.


In [17]:
from langchain_community.vectorstores import FAISS

# FAISS 벡터 스토어 생성 (문서 + 임베딩 모델)
vectorstore = FAISS.from_documents(docs_lc, embeddings)
# as_retriever(): 벡터 스토어를 Retriever 인터페이스로 감싸기
vector_retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

# 벡터 검색 실행
results = vector_retriever.invoke('딥러닝 모델 구조')
for doc in results:
    print(f"[{doc.metadata['index']}] {doc.page_content[:50]}")

[6] 트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 모델 구조입니다.
[3] GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행합니다.
[4] RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다.


---
## Step 10: EnsembleRetriever -- LangChain의 하이브리드 검색

**앙상블(Ensemble)**: 여러 모델의 결과를 합쳐서 더 나은 결과를 만드는 기법.  
머신러닝에서 '한 명의 천재보다 여러 명의 보통 사람이 모이면 더 나은 판단을 한다'는 아이디어.

- `weights=[0.5, 0.5]`: 벡터 검색 50% + BM25 검색 50% 비중
- 가중치를 조절하여 검색 성격을 바꿀 수 있음 (e.g., 법률 문서 -> BM25 비중 높이기)

In [18]:
# EnsembleRetriever import
# 주의: LangChain 버전에 따라 import 경로가 다를 수 있음
#   - langchain.retrievers (구버전)
#   - langchain_community.retrievers (최신)
#   - langchain_classic.retrievers (일부 버전)
# 에러 나면 다른 경로로 시도해보세요!
try:
    from langchain.retrievers import EnsembleRetriever
except ImportError:
    try:
        from langchain_community.retrievers import EnsembleRetriever
    except ImportError:
        from langchain_classic.retrievers import EnsembleRetriever

# 벡터 + BM25 앙상블 (50:50 가중치)
ensemble = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5],  # [벡터 비중, BM25 비중]
)

results = ensemble.invoke("Python 데이터 과학")
for doc in results:
    print(f"[{doc.metadata['index']}] {doc.page_content[:50]}")

[0] Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.
[9] 임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.
[2] 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니다.
[8] 프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.


In [19]:
# ════════════════════════════════════════════════════════════════
# 가중치 실험: 벡터 비중 vs BM25 비중을 바꿔가며 결과 비교
# ════════════════════════════════════════════════════════════════
for w_vec, w_bm25 in [(0.2, 0.8), (0.5, 0.5), (0.8, 0.2)]:
    ensemble = EnsembleRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        weights=[w_vec, w_bm25],
    )
    results = ensemble.invoke("Python 데이터 과학")
    top_idx = results[0].metadata['index']
    print(f"BM25={w_bm25}, Vector={w_vec} -> Top-1: [{top_idx}] {results[0].page_content[:40]}")

BM25=0.8, Vector=0.2 -> Top-1: [0] Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니
BM25=0.5, Vector=0.5 -> Top-1: [0] Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니
BM25=0.2, Vector=0.8 -> Top-1: [0] Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니


---
## Step 11: 세 가지 검색 방법 비교 실험

BM25, 벡터 검색, 앙상블 검색의 Top-3 결과를 나란히 비교합니다.  
어떤 쿼리에서 어떤 방법이 더 나은 결과를 보이는지 관찰해보세요.

In [20]:
# 앙상블 재초기화 (50:50)
ensemble = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5],
)

queries = ["Python 프로그래밍 언어", "딥러닝 모델 구조", "FAISS 라이브러리"]
for q in queries:
    bm25_res = bm25_retriever.invoke(q)
    vec_res = vector_retriever.invoke(q)
    ens_res = ensemble.invoke(q)

    bm25_ids = [d.metadata['index'] for d in bm25_res]
    vec_ids = [d.metadata['index'] for d in vec_res]
    ens_ids = [d.metadata['index'] for d in ens_res][:3]  # top-3만

    print(f"Query: {q}")
    print(f"  BM25:     {bm25_ids}")
    print(f"  Vector:   {vec_ids}")
    print(f"  Ensemble: {ens_ids}")
    print()

Query: Python 프로그래밍 언어
  BM25:     [0, 3, 8]
  Vector:   [0, 1, 3]
  Ensemble: [0, 3, 1]

Query: 딥러닝 모델 구조
  BM25:     [6, 9, 8]
  Vector:   [6, 3, 4]
  Ensemble: [6, 3, 9]

Query: FAISS 라이브러리
  BM25:     [9, 8, 7]
  Vector:   [7, 4, 2]
  Ensemble: [7, 9, 4]



---
## 정리

| 방법 | 원리 | 장점 | 단점 |
|------|------|------|------|
| 키워드 검색 | 단어 일치 여부 | 빠름, 고유명사에 강함 | 유의어 못 찾음, 조사 문제 |
| 벡터 검색 | 임베딩 공간 유사도 | 의미적 유사 문서 발견 | 드문 고유명사에 약함 |
| TF-IDF | 단어 빈도 + 희귀도 | 키워드 중요도 반영 | 문서 길이/TF 포화 미보정 |
| BM25 | TF-IDF + k1/b 보정 | 실전 키워드 검색 표준 | 의미적 유사성 못 봄 |
| 하이브리드 | 키워드 + 벡터 합산 | 두 방법의 장점 결합 | 점수 스케일 맞추기 필요 |

### 다음 시간 예고
- 두 검색기의 **점수 스케일이 다른 문제**를 해결하는 방법 (RRF 등)
- 더 공정한 앙상블 방법 학습 예정